# 🚀 POLYDIM V5.1 - EL FIN DE LOS ATRACTORES PERIÓDICOS
Implementa la matriz de adyacencia paramétrica (Top-K aprendible) estrictamente causal.
Introduce una fase asimétrica en la salida para romper ciclos.

In [1]:
# ============================================================
# POLYDIM V5.1 - SPARSE MAGNETIC TOP-K (ROMPIENDO ATRACTORES)
# ============================================================
# Implementación del plan V5.1 para evitar ciclos de generación.
# Cambios:
# 1. SparseMagneticLaplacian: En lugar de un grafo fijo simetrizado,
#    usamos una matriz aprendible W_adj estrictamente causal, ruteada por Top-K.
# 2. Fase magnética Theta es paramétrica y aprendible.
# 3. Fase de Salida: Se inyecta un parámetro Gamma para romper simetría
#    antes del colapso C -> R.

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import math
import requests
import time
import json

torch.manual_seed(1337)
np.random.seed(1337)

# -----------------------------------------------------------
# 0. DATASET
# -----------------------------------------------------------
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
text = requests.get(url).text
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(f"[DATA] Longitud: {len(text)}, Vocab: {vocab_size}")

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]

# -----------------------------------------------------------
# 1. CORE COMPLEJO (V5.1)
# -----------------------------------------------------------

class ComplexLinear(nn.Module):
    def __init__(self, in_features, out_features, bias=True):
        super().__init__()
        self.fc_r = nn.Linear(in_features, out_features, bias=bias)
        self.fc_i = nn.Linear(in_features, out_features, bias=bias)

    def forward(self, x):
        real = self.fc_r(x.real) - self.fc_i(x.imag)
        imag = self.fc_r(x.imag) + self.fc_i(x.real)
        return torch.complex(real, imag)

class ComplexLayerNorm(nn.Module):
    def __init__(self, D):
        super().__init__()
        self.ln_r = nn.LayerNorm(D)
        self.ln_i = nn.LayerNorm(D)

    def forward(self, x):
        return torch.complex(self.ln_r(x.real), self.ln_i(x.imag))

class ComplexGELU(nn.Module):
    def forward(self, x):
        return torch.complex(F.gelu(x.real), F.gelu(x.imag))

class ComplexDropout(nn.Module):
    def __init__(self, p=0.1):
        super().__init__()
        self.p = p
    def forward(self, x):
        if self.training:
            mask = (torch.rand(x.shape, device=x.device) > self.p).float() / (1.0 - self.p)
            return x * mask
        return x

# -----------------------------------------------------------
# 2. COMPONENTES POLYDIM (V5.1)
# -----------------------------------------------------------

class SparseMagneticLaplacian(nn.Module):
    """
    Laplaciano Magnético Dirigido (DAG) con Top-K aprendible.
    Rompe la hermiticidad para preservar causalidad pura.
    """
    def __init__(self, N: int, K: int = 4):
        super().__init__()
        self.N = N
        self.K = K

        # Conexiones topológicas y fase aprendibles
        self.W_adj = nn.Parameter(torch.randn(N, N) * 0.02)
        self.Theta = nn.Parameter(torch.randn(N, N) * 0.02)

        # Máscara causal (estrictamente inferior para no ver ni el presente ni futuro)
        mask = torch.tril(torch.ones(N, N), diagonal=-1)
        self.register_buffer('causal_mask', mask)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        T_seq = x.shape[-2]

        W = self.W_adj[:T_seq, :T_seq]
        mask = self.causal_mask[:T_seq, :T_seq]

        # Forzar causalidad
        adj_masked = W * mask

        # Top-K Routing
        if self.K >= T_seq:
            A_sparse = adj_masked
        else:
            topk_vals, topk_indices = torch.topk(adj_masked, self.K, dim=-1)
            A_sparse = torch.zeros_like(adj_masked)
            A_sparse.scatter_(-1, topk_indices, topk_vals)

        A_sparse = F.relu(A_sparse) # Pesos de arista positivos

        D_s = torch.diag(A_sparse.sum(dim=-1))

        Theta_masked = self.Theta[:T_seq, :T_seq] * mask
        H_q = A_sparse.to(torch.cfloat) * torch.exp(1j * Theta_masked)

        L_q = D_s.to(torch.cfloat) - H_q

        if not torch.is_complex(x):
            x = x.to(torch.cfloat)

        return torch.einsum('ij,...jd->...id', L_q, x)


class FourierEmbedding(nn.Module):
    def __init__(self, D: int, max_len: int = 5000):
        super().__init__()
        self.D = D
        freqs = torch.exp(torch.arange(0, D, 2).float() * (-math.log(10000.0) / D))
        self.freqs = nn.Parameter(freqs)

    def forward(self, positions: torch.Tensor) -> torch.Tensor:
        B, N = positions.shape
        angles = positions.unsqueeze(-1) * self.freqs.unsqueeze(0).unsqueeze(0)
        emb = torch.zeros(B, N, self.D, device=positions.device)
        emb[:, :, 0::2] = torch.sin(angles)
        emb[:, :, 1::2] = torch.cos(angles)
        return emb.to(torch.cfloat)


class IsometricRotation(nn.Module):
    def __init__(self, D: int):
        super().__init__()
        self.D = D
        self.phase = nn.Parameter(torch.randn(D) * 0.01)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if not torch.is_complex(x): x = x.to(torch.cfloat)
        x_f = torch.fft.fft(x, dim=-1)
        x_f = x_f * torch.exp(1j * self.phase).unsqueeze(0)
        x = torch.fft.ifft(x_f, dim=-1)
        return x


class PolydimLayerV5(nn.Module):
    def __init__(self, D: int, N: int, n_nodes: int, K_skip: int, dropout: float):
        super().__init__()
        self.D = D
        self.n_nodes = n_nodes
        self.d_node = D // n_nodes

        self.norm1 = ComplexLayerNorm(D)
        self.norm2 = ComplexLayerNorm(D)

        self.rotors = nn.ModuleList([
            IsometricRotation(self.d_node) for _ in range(n_nodes)
        ])

        self.laplacian = SparseMagneticLaplacian(N, K=K_skip)

        self.mlp = nn.Sequential(
            ComplexLinear(D, 4 * D),
            ComplexGELU(),
            ComplexDropout(dropout),
            ComplexLinear(4 * D, D),
            ComplexDropout(dropout)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, seq_len, D = x.shape
        residual = x
        x = self.norm1(x)

        x_nodes = x.reshape(B, seq_len, self.n_nodes, self.d_node)
        rotated = []
        for i, rotor in enumerate(self.rotors):
            rotated.append(rotor(x_nodes[:, :, i, :]))

        x = torch.stack(rotated, dim=2).reshape(B, seq_len, D)
        x = self.laplacian(x)

        x = residual + x
        residual = x
        x = self.norm2(x)
        x = self.mlp(x)
        x = residual + x
        return x


class PolydimMotorV5_1(nn.Module):
    def __init__(
        self,
        vocab_size: int = 50000,
        D: int = 256,
        N: int = 128,
        n_layers: int = 4,
        n_nodes: int = 4,
        K_skip: int = 16,
        dropout: float = 0.1
    ):
        super().__init__()
        self.vocab_size = vocab_size
        self.D = D

        self.token_embed = nn.Embedding(vocab_size, D)
        self.pos_embed = FourierEmbedding(D, max_len=N)
        self.dropout = ComplexDropout(dropout)

        self.layers = nn.ModuleList([
            PolydimLayerV5(D, N, n_nodes, K_skip, dropout)
            for _ in range(n_layers)
        ])

        self.norm = ComplexLayerNorm(D)

        # Fase de ruptura de simetría (Disipación final)
        self.gamma_phase = nn.Parameter(torch.randn(D) * 0.1)

        self.to_vocab = nn.Linear(D * 2, vocab_size)

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        B, seq_len = input_ids.shape
        token_emb = self.token_embed(input_ids).to(torch.cfloat)

        pos_ids = torch.arange(seq_len, device=input_ids.device).unsqueeze(0).expand(B, -1)
        pos_emb = self.pos_embed(pos_ids)

        x = self.dropout(token_emb + pos_emb)

        for layer in self.layers:
            x = layer(x)

        x = self.norm(x)

        # Aplicamos la fase de ruptura
        x = x * torch.exp(1j * self.gamma_phase).unsqueeze(0).unsqueeze(0)

        x_out = torch.cat([x.real, x.imag], dim=-1)
        logits = self.to_vocab(x_out)

        return logits


# -----------------------------------------------------------
# 3. ENTRENAMIENTO CONFIG
# -----------------------------------------------------------

CONFIG = {
    "D": 256,
    "N": 128,
    "n_layers": 4,
    "n_nodes": 4,
    "K_skip": 16, # El modelo elegirá las 16 conexiones más fuertes hacia el pasado
    "dropout": 0.1,
    "batch_size": 32,
    "block_size": 128,
    "max_iters": 5000,
    "eval_interval": 500,
    "learning_rate": 1e-3,
    "device": 'cuda' if torch.cuda.is_available() else 'cpu'
}

print(f"\n[V5.1] Entrenando en: {CONFIG['device']}")

def get_batch(split):
    d = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - CONFIG['block_size'], (CONFIG['batch_size'],))
    x = torch.stack([d[i:i+CONFIG['block_size']] for i in ix])
    y = torch.stack([d[i+1:i+CONFIG['block_size']+1] for i in ix])
    return x.to(CONFIG['device']), y.to(CONFIG['device'])

@torch.no_grad()
def estimate_loss(model):
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(50)
        for k in range(50):
            X, Y = get_batch(split)
            logits = model(X)
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B*T, C), Y.view(B*T))
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

# -----------------------------------------------------------
# 4. RUN V5.1
# -----------------------------------------------------------
print("\n🚀 INICIANDO POLYDIM V5.1 (Sparse Magnetic DAG + Salida Disipativa)...")
model_v5 = PolydimMotorV5_1(
    vocab_size=vocab_size, D=CONFIG['D'], N=CONFIG['N'],
    n_layers=CONFIG['n_layers'], n_nodes=CONFIG['n_nodes'],
    K_skip=CONFIG['K_skip'], dropout=CONFIG['dropout']
).to(CONFIG['device'])

optimizer = torch.optim.AdamW(model_v5.parameters(), lr=CONFIG['learning_rate'])

t0 = time.time()
for iter in range(CONFIG['max_iters'] + 1):
    if iter % CONFIG['eval_interval'] == 0:
        losses = estimate_loss(model_v5)
        print(f"Step {iter:5d} | train_loss {losses['train']:.4f} | val_loss {losses['val']:.4f} | time {time.time()-t0:.1f}s")

    xb, yb = get_batch('train')
    logits = model_v5(xb)
    loss = F.cross_entropy(logits.view(-1, vocab_size), yb.view(-1))

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print("✅ Entrenamiento V5.1 finalizado.")

print("\n🧠 GENERANDO TEXTO CAUSAL V5.1...")
context = torch.zeros((1, 1), dtype=torch.long, device=CONFIG['device'])
model_v5.eval()
generated = []
for _ in range(500):
    cond = context[:, -CONFIG['block_size']:]
    logits = model_v5(cond)
    probs = F.softmax(logits[:, -1, :], dim=-1)
    next_tok = torch.multinomial(probs, num_samples=1)
    context = torch.cat((context, next_tok), dim=1)
    generated.append(next_tok.item())

print("\n--- V5.1 GENERADO ---")
print(decode(generated))

[DATA] Longitud: 1115394, Vocab: 65

[V5.1] Entrenando en: cuda

🚀 INICIANDO POLYDIM V5.1 (Sparse Magnetic DAG + Salida Disipativa)...
Step     0 | train_loss 4.3423 | val_loss 4.3383 | time 5.4s
Step   500 | train_loss 2.2106 | val_loss 2.2797 | time 79.6s
Step  1000 | train_loss 1.9834 | val_loss 2.1124 | time 154.5s
Step  1500 | train_loss 1.9092 | val_loss 2.0572 | time 229.7s
Step  2000 | train_loss 1.8585 | val_loss 2.0257 | time 304.9s
Step  2500 | train_loss 1.8342 | val_loss 1.9968 | time 379.7s
Step  3000 | train_loss 1.8179 | val_loss 1.9883 | time 454.4s
Step  3500 | train_loss 1.7917 | val_loss 1.9649 | time 529.1s
Step  4000 | train_loss 1.7791 | val_loss 1.9637 | time 604.1s
Step  4500 | train_loss 1.7752 | val_loss 1.9348 | time 679.3s
Step  5000 | train_loss 1.7663 | val_loss 1.9414 | time 754.5s
✅ Entrenamiento V5.1 finalizado.

🧠 GENERANDO TEXTO CAUSAL V5.1...

--- V5.1 GENERADO ---
Goure t, those cose threat. a l go the quise.
I have to thy dave a l accose, if we da